In [1]:
!pip install sentence-transformers -q
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import json
import os

In [2]:
!git clone -b main https://github.com/AsyaPradana/CBR-Narkotika-PN-Tulungagung.git

Cloning into 'CBR-Narkotika-PN-Tulungagung'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 83 (delta 20), reused 67 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 1.59 MiB | 1.63 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [3]:
DATA_PATH = "/content/CBR-Narkotika-PN-Tulungagung/data/processed/cases.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)

df.head()

(50, 9)


,case_id,jenis_perkara,no_perkara,pasal,terdakwa,dakwaan,putusan,jumlah_kata,text_full
0,1,Pidana Narkotika,102 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 148 ; pasal 112 ; pasal 132 ; pasal 193 ...,adila resti sari binti boimain h2,yang diajukan oleh penuntut umum yang pada po...,perkara pidana dengan acara pemeriksaan biasa...,8000,gnomor 102 pid sus 2025 pn tlg demi keadilan b...
1,2,Pidana Narkotika,104 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 112 ; pasal 148 ; pasal 114,zelly yusando alias nsincan bin rismaji 2,NaN,perkara pidana dengan acara pemeriksaan biasa...,7319,direktori utusan mahkamah agung republikp iidn...
2,3,Pidana Narkotika,105 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 148 ; pasal 112 ; pasal 101 ; pasal 13 ;...,riki suhendra bin sumarno 2,yang diajukan oleh penuntut umum ya ng pada p...,perkara pidana dengan agcara pemeriksaan bias...,10522,nomor 105 pid sus 2025 pn tlg demi keadilan be...
3,4,Pidana Narkotika,106 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 148 ; pasal 112 ; pasal 132 ; pasal 193 ...,wiji santoso bin satimin h2,yang diajukan oleh penuntut umum yang pada pe...,perkara pidana dengan acara pemeriksaan biasa...,10152,gnomor 106 pid sus 2025 pn tlg demi keadilan b...
4,5,Pidana Narkotika,107 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 112 ; pasal 127; pasal 132 ; pasal 132; ...,novel saputra ahmad dhani bin sonpan sopyan 2,yang diajukan oleh penuntut umum yang pada po...,perkara pidana dengan acara pemeriksaan biasa...,9426,direktori putusan mahkamah agung re lik indone...


In [4]:
documents = df["text_full"].fillna("").tolist()

print("Jumlah dokumen:", len(documents))

Jumlah dokumen: 50


In [5]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words=None
)

tfidf_matrix = vectorizer.fit_transform(documents)

print(tfidf_matrix.shape)

(50, 5000)


In [11]:
def retrieve(query, k=5):

    query_vec = vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vec,
        tfidf_matrix
    ).flatten()

    top_idx = similarities.argsort()[::-1][:k]

    hasil = []

    for idx in top_idx:

        hasil.append({
            "case_id": int(df.iloc[idx]["case_id"]),
            "similarity": round(float(similarities[idx]),4),
            "terdakwa": str(df.iloc[idx]["terdakwa"])[:50]
        })

    return hasil

In [12]:
query = """
Terdakwa menjual sabu kepada pembeli.
Barang bukti berupa sabu 0.5 gram.
Melanggar pasal 114 UU Narkotika.
"""

In [13]:
retrieve(query)

[{'case_id': 42,
  'similarity': 0.4224,
  'terdakwa': 'rimba bagus arengga aklias rimbuk bin sutaji 2'},
 {'case_id': 41,
  'similarity': 0.3274,
  'terdakwa': 'tus ifngal khoir alias peno bin sutrisno 2'},
 {'case_id': 19,
  'similarity': 0.2874,
  'terdakwa': 'mochamad david nurul fu and als grandong bin nur c'},
 {'case_id': 40,
  'similarity': 0.2758,
  'terdakwa': 'richard edwardo sutrisno alias edo 2'},
 {'case_id': 11, 'similarity': 0.2659, 'terdakwa': 'ribut bin padi 2'}]

In [17]:
hasil = retrieve(query, k=5)

for h in hasil:

    print("="*50)

    print("Case ID:", h["case_id"])

    no_perkara = df[df['case_id'] == h['case_id']]['no_perkara'].iloc[0]
    print("Nomor:", no_perkara)

    print("Similarity:", round(h["similarity"],4))

Case ID: 42
Nomor: 77 pid sus 2025 pn tlg demi keadilan berdasarkan ketuhanan yang maha esa pengadilan negeri tulungagung yang mengadili perkara pidana dengan acara pemeriksaan biasa dalam tingkat pertama menjatuhkan aputusan sebagai berikut dalam perkara para terdakwa h1 nama lengkap rimba bagus arengga aklias rimbuk bin sutaji 2 tempat lahir tulungagung 3 umur tanggal lahir 20 tahun 15 agustus 2004 4 jenis kelamin laki laki 5 kebangsaan indonesia 6 tempat tinggal dsn tanggul rt 04 rw 02 ds tanggul turus keec besuki kab tulungagung 7 agama islam 8 pekerjaan wiraswasta terdakwa rimbag bagus arengga alias rimbuk bin sutaji ditangkap tanggal 27 februari 2025 terdakwa rimba bagus arengga alias rimbuk bin sutaji ditahan di lembaga pemasyarakatan tulungagung oleh 1 penyidik sejak tanggal 28 februari 2025 sampai dengan tanggal 19 maret a2025 2 penyidik perpanjangan oleh penuntut umum sejak taniggal 20 maret h2025 sampai dengan tanggal 14 april 2025 3 penuntut umum sejak tanggal 15 april 2025

In [15]:
df.head(3)

,case_id,jenis_perkara,no_perkara,pasal,terdakwa,dakwaan,putusan,jumlah_kata,text_full
0,1,Pidana Narkotika,102 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 148 ; pasal 112 ; pasal 132 ; pasal 193 ...,adila resti sari binti boimain h2,yang diajukan oleh penuntut umum yang pada po...,perkara pidana dengan acara pemeriksaan biasa...,8000,gnomor 102 pid sus 2025 pn tlg demi keadilan b...
1,2,Pidana Narkotika,104 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 112 ; pasal 148 ; pasal 114,zelly yusando alias nsincan bin rismaji 2,NaN,perkara pidana dengan acara pemeriksaan biasa...,7319,direktori utusan mahkamah agung republikp iidn...
2,3,Pidana Narkotika,105 pid sus 2025 pn tlg demi keadilan berdasar...,pasal 148 ; pasal 112 ; pasal 101 ; pasal 13 ;...,riki suhendra bin sumarno 2,yang diajukan oleh penuntut umum ya ng pada p...,perkara pidana dengan agcara pemeriksaan bias...,10522,nomor 105 pid sus 2025 pn tlg demi keadilan be...


In [18]:
eval_path = "/content/CBR-Narkotika-PN-Tulungagung/data/eval"

os.makedirs(eval_path, exist_ok=True)

In [19]:
queries = [
    {
        "query_id":1,
        "query":"Terdakwa menjual sabu",
        "ground_truth":[1]
    },

    {
        "query_id":2,
        "query":"Penyalahgunaan narkotika golongan I",
        "ground_truth":[2]
    },

    {
        "query_id":3,
        "query":"Memiliki sabu tanpa izin",
        "ground_truth":[3]
    },

    {
        "query_id":4,
        "query":"Perantara jual beli sabu",
        "ground_truth":[4]
    },

    {
        "query_id":5,
        "query":"Menyimpan narkotika golongan I",
        "ground_truth":[5]
    }
]

In [20]:
with open(
    f"{eval_path}/queries.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        queries,
        f,
        ensure_ascii=False,
        indent=4
    )

print("queries.json berhasil dibuat")

queries.json berhasil dibuat


In [21]:
for q in queries:

    hasil = retrieve(
        q["query"],
        k=3
    )

    print("\nQUERY:")
    print(q["query"])

    print("\nTOP 3:")

    for h in hasil:

        print(
            h["case_id"],
            round(h["similarity"],3)
        )

    print("-"*50)


QUERY:
Terdakwa menjual sabu

TOP 3:
42 0.517
19 0.384
41 0.373
--------------------------------------------------

QUERY:
Penyalahgunaan narkotika golongan I

TOP 3:
26 0.109
12 0.094
36 0.087
--------------------------------------------------

QUERY:
Memiliki sabu tanpa izin

TOP 3:
42 0.338
41 0.223
11 0.18
--------------------------------------------------

QUERY:
Perantara jual beli sabu

TOP 3:
42 0.343
41 0.232
19 0.179
--------------------------------------------------

QUERY:
Menyimpan narkotika golongan I

TOP 3:
26 0.145
12 0.124
36 0.117
--------------------------------------------------


#IndoBERT Embedding

In [23]:
!pip install sentence-transformers transformers torch -q
from sentence_transformers import SentenceTransformer

In [24]:
model = SentenceTransformer(
    "indobenchmark/indobert-base-p1"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [25]:
documents = df["text_full"].fillna("").tolist()

embeddings = model.encode(
    documents,
    show_progress_bar=True
)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [26]:
print(embeddings.shape)

(50, 768)


In [27]:
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
def retrieve_bert(query, k=5):

    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    top_idx = similarities.argsort()[::-1][:k]

    results = []

    for idx in top_idx:

        results.append({
            "case_id": int(df.iloc[idx]["case_id"]),
            "similarity": round(
                float(similarities[idx]),
                4
            )
        })

    return results

In [29]:
query = """
Terdakwa menjadi perantara jual beli sabu
"""

In [30]:
retrieve_bert(query)

[{'case_id': 37, 'similarity': 0.3592},
 {'case_id': 46, 'similarity': 0.3502},
 {'case_id': 36, 'similarity': 0.3482},
 {'case_id': 26, 'similarity': 0.3442},
 {'case_id': 41, 'similarity': 0.3437}]

In [31]:
print("=== TF-IDF ===")
print(retrieve(query))

print("\n=== IndoBERT ===")
print(retrieve_bert(query))

=== TF-IDF ===
[{'case_id': 42, 'similarity': 0.3949, 'terdakwa': 'rimba bagus arengga aklias rimbuk bin sutaji 2'}, {'case_id': 41, 'similarity': 0.3027, 'terdakwa': 'tus ifngal khoir alias peno bin sutrisno 2'}, {'case_id': 19, 'similarity': 0.2917, 'terdakwa': 'mochamad david nurul fu and als grandong bin nur c'}, {'case_id': 9, 'similarity': 0.2626, 'terdakwa': 'yoni wibakso bin alm mukani 2'}, {'case_id': 40, 'similarity': 0.2561, 'terdakwa': 'richard edwardo sutrisno alias edo 2'}]

=== IndoBERT ===
[{'case_id': 37, 'similarity': 0.3592}, {'case_id': 46, 'similarity': 0.3502}, {'case_id': 36, 'similarity': 0.3482}, {'case_id': 26, 'similarity': 0.3442}, {'case_id': 41, 'similarity': 0.3437}]


In [32]:
!mkdir -p /content/CBR-Narkotika-PN-Tulungagung/data/eval